# Multi-Agent Podcast Research Pipeline

In the previous notebook you built an agent from scratch: chat history, tool calling, the agent loop, and skills. Now we hand the loop to **CrewAI** and focus on what frameworks unlock: **multi-agent orchestration**.

| Step | What You Already Know | CrewAI Equivalent |
|------|----------------------|-------------------|
| — | `system_prompt` / skill | `Agent(role, goal, backstory)` |
| — | `execute_search()` | `EXASearchTool()` |
| — | The `while` loop | CrewAI execution engine |
| — | "Research this" | `Task(description, expected_output)` |

| Step | New Concept | What It Unlocks |
|------|------------|----------------|
| 1 | **Agent + Task + Tool** | Map your hand-rolled code to CrewAI in one shot |
| 2 | **Parameters** | Bound iteration count, rate limits, timeouts |
| 3 | **Second Agent + Context** | One agent's output feeds the next — the core multi-agent pattern |
| 4 | **Crew** | Orchestrate a sequential pipeline with one API call |
| 5 | **Dynamic Inputs** | Parametrize topic, month, year for reuse |


In [6]:
# !pip install -q "crewai[tools]" python-dotenv

import os
from datetime import datetime

from crewai import Agent, Crew, Process, Task
from crewai.llm import LLM
from crewai_tools import EXASearchTool
from dotenv import load_dotenv

load_dotenv("../../.env")

llm = LLM(
    model="openai/gpt-4.1-2025-04-14",
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1",
)

os.environ["EXA_API_KEY"] = os.getenv("EXA_API_KEY", "")
search_tool = EXASearchTool()


---

## Step 1: Agent + Task + Tool — The CrewAI Mapping

You already built all three primitives by hand. CrewAI wraps them:

- Your **skill / system prompt** becomes `Agent(role=..., goal=..., backstory=...)`
- Your **search function** becomes a pre-built tool: `EXASearchTool()`
- Your **"research this" instruction** becomes `Task(description=..., expected_output=...)`

> **Prompt Engineering — Role/Persona Priming:** `backstory` is the same technique as the `system_prompt` you wrote by hand — it sets the agent's identity, tone, and focus.
> **Prompt Engineering — Specificity / Constraint Engineering:** `Task(description=..., expected_output=...)` is where constraints live. The more specific these are, the better the output quality.

Let's wire them up and run one agent to confirm the mapping.


In [7]:
researcher = Agent(
    role="AI Podcast Researcher",
    goal="Find recent releases, articles, topics, and progress about AI LLMs",
    # Prompt Engineering: Role/Persona Priming
    backstory=(
        "You are a seasoned researcher with a knack for uncovering the latest developments. "
        "You find the most relevant information and present it clearly."
    ),
    llm=llm,
    tools=[search_tool],
    verbose=True,
)

# Prompt Engineering: Specificity / Constraint Engineering
research_task = Task(
    description=(
        "Create a research brief for a podcast episode about Google I/O 2026. "
        "Search for 5 recent and important developments. "
        "For each, include: title, short summary, and why it matters for listeners."
    ),
    expected_output="A podcast research brief with 5 findings, each with title, summary, and listener angle.",
    agent=researcher,
)

single_agent_crew = Crew(
    agents=[researcher],
    tasks=[research_task],
    process=Process.sequential,
    verbose=True,
)

single_result = single_agent_crew.kickoff()
single_result


╭──────────────────────────────────────────── Crew Execution Started ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 1a590300-147f-4c84-8893-2ee95052a10a                                                                       │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: AI Podcast Researcher                                                                                   │
│                                                                                                                 │
│  Task: Create a research brief for a podcast episode about Google I/O 2026. Search for 5 recent and important   │
│  developments. For each, include: title, short summary, and why it matters for listeners.                       │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Output()

╭──────────────────────────────────────────── 🔧 Agent Tool Execution ────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: AI Podcast Researcher                                                                                   │
│                                                                                                                 │
│  Thought: Thought: I need to search for recent and important developments announced at or related to Google     │
│  I/O 2026, with a focus on impactful or innovative topics that would matter to podcast listeners.               │
│                                                                                                                 │
│  Using Tool: EXASearchTool                                                                                      │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────────── Tool Input ───────────────────────────────────────────────────╮
│                                                                                                                 │
│  "{\"search_query\": \"Google I/O 2026 highlights announcements\", \"start_published_date\": \"2026-05-01\",    │
│  \"end_published_date\": \"2026-06-01\"}"                                                                       │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: AI Podcast Researcher                                                                                   │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ---                                                                                                            │
│  **Podcast Research Brief: Google I/O 2026 – The 5 Biggest Developments**                                       │
│                                                                                                                 │
│  1. **Gemini 3.5: The Next Leap in AI Integration**                                                             │
│     - **Summary:** Google unveiled Gemini 3.5, its new generation large language model that powers everything   │
│  from Google Search to Workspace and Android. With a focus on multimodal understanding, Gemini 3.5 can          │
│  interpret images, videos, and text seamlessly, while providing more accurate, contextual responses to user     │
│  queries.                                                                                                       │
│     - **Why it matters:** Listeners will feel the impact of Gemini 3.5 in everyday life via smarter             │
│  assistants, more intuitive search results, and AI-powered suggestions across Google products. It signals the   │
│  continued mainstreaming of advanced AI in daily tasks and raises questions for consumers about privacy,        │
│  creativity, and the limits of automated reasoning.                                                             │
│     - **Sources:** [The                                                                                         │
│  Verge](https://www.theverge.com/tech/933415/google-io-2026-biggest-announcements-ai-gemini),                   │
│  [TheNeurOn](https://www.theneuron.ai/explainer-articles/google-io-2026-was-about-turning-every-google-app-int  │
│  o-an-agent/)                                                                                                   │
│                                                                                                                 │
│  2. **AI Agents for Every Google App**                                                                          │
│     - **Summary:** Google announced the rollout of ‘AI agents’ across its entire suite of applications,         │
│  transforming products like Gmail, Google Maps, and Docs into proactive, conversational helpers. These agents   │
│  can book reservations, automate repetitive tasks, and even handle complex multi-step workflows.                │
│     - **Why it matters:** Listeners should prepare for a paradigm shift in how they use technology: instead of  │
│  passive tools, apps now actively anticipate and serve users’ needs, potentially saving hours each week and     │
│  opening new doors for accessibility and productivity.                                                          │
│     - **Sources:**                                                                                              │
│  [TheNeurOn](https://www.theneuron.ai/explainer-articles/google-io-2026-was-about-turning-every-google-app-int  │
│  o-an-agent/), [Google                                                                                          │
│  Blog](https://blog.google/innovation-and-ai/technology/ai/google-io-2026-all-our-announcements/)               │
│                                                                                                                 │
│  3. **Antigravity UI and ‘Project Spark’: Redefining th

╭──────────────────────────────────────────────── Task Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: c3a1894d-a725-4a83-bb42-fb430fa1c8f8                                                                     │
│  Agent: AI Podcast Researcher                                                                                   │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 1a590300-147f-4c84-8893-2ee95052a10a                                                                       │
│  Tool Args:                                                                                                     │
│  Final Output: ---                                                                                              │
│  **Podcast Research Brief: Google I/O 2026 – The 5 Biggest Developments**                                       │
│                                                                                                                 │
│  1. **Gemini 3.5: The Next Leap in AI Integration**                                                             │
│     - **Summary:** Google unveiled Gemini 3.5, its new generation large language model that powers everything   │
│  from Google Search to Workspace and Android. With a focus on multimodal understanding, Gemini 3.5 can          │
│  interpret images, videos, and text seamlessly, while providing more accurate, contextual responses to user     │
│  queries.                                                                                                       │
│     - **Why it matters:** Listeners will feel the impact of Gemini 3.5 in everyday life via smarter             │
│  assistants, more intuitive search results, and AI-powered suggestions across Google products. It signals the   │
│  continued mainstreaming of advanced AI in daily tasks and raises questions for consumers about privacy,        │
│  creativity, and the limits of automated reasoning.                                                             │
│     - **Sources:** [The                                                                                         │
│  Verge](https://www.theverge.com/tech/933415/google-io-2026-biggest-announcements-ai-gemini),                   │
│  [TheNeurOn](https://www.theneuron.ai/explainer-articles/google-io-2026-was-about-turning-every-google-app-int  │
│  o-an-agent/)                                                                                                   │
│                                                                                                                 │
│  2. **AI Agents for Every Google App**                                                                          │
│     - **Summary:** Google announced the rollout of ‘AI agents’ across its entire suite of applications,         │
│  transforming products like Gmail, Google Maps, and Docs into proactive, conversational helpers. These agents   │
│  can book reservations, automate repetitive tasks, and even handle complex multi-step workflows.                │
│     - **Why it matters:** Listeners should prepare for a paradigm shift in how they use technology: instead of  │
│  passive tools, apps now actively anticipate and serve users’ needs, potentially saving hours each week and     │
│  opening new doors for accessibility and productivity.                                                          │
│     - **Sources:**                                                                                              │
│  [TheNeurOn](https://www.theneuron.ai/explainer-articles/google-io-2026-was-about-turning-every-google-app-int  │
│  o-an-agent/), [Google                                                                                          │
│  Blog](https://blog.google/innovation-and-ai/technology/ai/google-io-2026-all-our-announcements/)               │
│                                                       

╭─────────────────────────────────────────── Trace Batch Finalization ────────────────────────────────────────────╮
│ ✅ Trace batch finalized with session ID: 938c0d09-1f01-4a01-a265-ab569a08e69e                                  │
│                                                                                                                 │
│ 🔗 View here:                                                                                                   │
│ https://app.crewai.com/crewai_plus/ephemeral_trace_batches/938c0d09-1f01-4a01-a265-ab569a08e69e?access_code=TRA │
│ CE-d18dee7d40                                                                                                   │
│ 🔑 Access Code: TRACE-d18dee7d40                                                                                │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

CrewOutput(raw='---\n**Podcast Research Brief: Google I/O 2026 – The 5 Biggest Developments**\n\n1. **Gemini 3.5: The Next Leap in AI Integration**\n   - **Summary:** Google unveiled Gemini 3.5, its new generation large language model that powers everything from Google Search to Workspace and Android. With a focus on multimodal understanding, Gemini 3.5 can interpret images, videos, and text seamlessly, while providing more accurate, contextual responses to user queries.\n   - **Why it matters:** Listeners will feel the impact of Gemini 3.5 in everyday life via smarter assistants, more intuitive search results, and AI-powered suggestions across Google products. It signals the continued mainstreaming of advanced AI in daily tasks and raises questions for consumers about privacy, creativity, and the limits of automated reasoning.\n   - **Sources:** [The Verge](https://www.theverge.com/tech/933415/google-io-2026-biggest-announcements-ai-gemini), [TheNeurOn](https://www.theneuron.ai/explai

That is the full mapping. CrewAI handled the loop, tool dispatch, and history for you. The agent searched the web, decided when it had enough, and returned text — exactly like the loop you built by hand.

---

## Step 2: Production Parameters

Your hand-rolled loop had `max_iterations`. CrewAI gives you finer controls.


In [8]:
researcher = Agent(
    role="AI Podcast Researcher",
    goal="Find recent releases, articles, topics, and progress about {topic}",
    backstory=(
        "You are a seasoned researcher with a knack for uncovering the latest developments. "
        "You find the most relevant information and present it clearly."
    ),
    llm=llm,
    tools=[search_tool],
    verbose=True,
    max_iter=5,
    max_rpm=12,
    max_execution_time=300,
    respect_context_window=True,
)

print(f"Researcher ready: max_iter={researcher.max_iter}, max_rpm={researcher.max_rpm}")


Researcher ready: max_iter=5, max_rpm=12


---

## Step 3: Second Agent + Context — The Multi-Agent Pattern

This is where CrewAI earns its keep. In your hand-rolled loop you had one agent. Now we add a **reporting analyst** whose task receives the researcher's output via `context`.

This is the equivalent of piping one agent's final answer into another agent's system prompt — except CrewAI handles the plumbing.

> **Prompt Engineering — Prompt Chaining:** `context=[research_task]` is the CrewAI implementation of prompt chaining — one prompt's output becomes the next prompt's input, sequentially building toward the final result.


In [9]:
reporting_analyst = Agent(
    role="AI Podcast Reporting Analyst",
    goal="Turn raw research findings about {topic} into a structured podcast briefing report",
    backstory=(
        "You are a meticulous analyst who turns messy research into structured reports. "
        "You organize findings into sections that are easy for podcast hosts to understand and riff on."
    ),
    llm=llm,
    verbose=True,
    max_iter=4,
    max_rpm=10,
    max_execution_time=240,
    respect_context_window=True,
)


In [10]:
research_task = Task(
    description=(
        "Conduct thorough podcast research about {topic}. "
        "Find recent releases, articles, products, debates, and progress. "
        "Use the current date context: {current_month} {current_year}."
    ),
    expected_output=(
        "A list of 10 relevant findings about {topic}. For each finding include: "
        "title, short summary, why it matters for listeners, and source if available."
    ),
    agent=researcher,
)

reporting_task = Task(
    description=(
        "Review the research findings and organize them into a complete podcast briefing report. "
        "Group related findings into sections and explain the narrative arc for the episode."
    ),
    expected_output=(
        "A markdown report with: episode angle, key sections with findings grouped thematically, "
        "important facts, listener takeaways, and suggested host questions."
    ),
    agent=reporting_analyst,
    context=[research_task],  # Prompt Chaining: research_task output feeds directly into this task
)


The key line is `context=[research_task]`. When CrewAI runs `reporting_task`, it automatically injects the full output of `research_task` into the reporting analyst's prompt. The analyst never saw the raw web search results — it only sees the researcher's curated brief.

---

## Step 4: The Crew — Orchestrate the Pipeline

Now we assemble both agents and both tasks into a `Crew`. CrewAI runs them sequentially: researcher first, then reporting analyst with context.


In [12]:
podcast_crew = Crew(
    agents=[researcher, reporting_analyst],
    tasks=[research_task, reporting_task],
    process=Process.sequential,
    verbose=True,
    cache=True,
    max_rpm=25,
)

inputs = {
    "topic": "Google I/O 2026 Top Announcements",
    "current_month": datetime.now().strftime("%B"),
    "current_year": str(datetime.now().year),
}

result = podcast_crew.kickoff(inputs=inputs)
result


╭──────────────────────────────────────────── Crew Execution Started ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 294b45e5-2d05-46a8-a988-78a4a5cf965f                                                                       │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: AI Podcast Researcher                                                                                   │
│                                                                                                                 │
│  Task: Conduct thorough podcast research about Google I/O 2026 Top Announcements. Find recent releases,         │
│  articles, products, debates, and progress. Use the current date context: May 2026.                             │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Output()

╭──────────────────────────────────────────── 🔧 Agent Tool Execution ────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: AI Podcast Researcher                                                                                   │
│                                                                                                                 │
│  Thought: Action: EXASearchTool                                                                                 │
│                                                                                                                 │
│  Using Tool: EXASearchTool                                                                                      │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────────── Tool Input ───────────────────────────────────────────────────╮
│                                                                                                                 │
│  "{\"search_query\": \"Google I/O 2026 top announcements\", \"start_published_date\": \"2026-05-01\",           │
│  \"end_published_date\": \"2026-05-31\", \"include_domains\": null}"                                            │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: AI Podcast Reporting Analyst                                                                            │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  # Podcast Briefing Report: “Google I/O 2026 – The Year Google Went ‘Agentic’: Top Announcements and What They  │
│  Mean for You”                                                                                                  │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ## Episode Angle                                                                                               │
│                                                                                                                 │
│  **“Google I/O 2026: The Leap Toward Agentic Apps and Everyday AI”**                                            │
│                                                                                                                 │
│  This year’s I/O wasn’t just about new gadgets or features—it signaled Google’s pivot to deeply integrated AI   │
│  agents, “ambient intelligence,” and a future where apps aren’t just tools, but partners in getting things      │
│  done. The announcement spread from flashy new hardware to subtle but major shifts in software, privacy, and    │
│  AI reliability.                                                                                                │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ## Key Sections & Thematic Groupings                                                                           │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ### 1. Google’s Vision & Core Themes                                                                           │
│                                                                                                                 │
│  - **Sundar Pichai’s Keynote Framing**                                                                          │
│    - Google’s “north stars”: Trustworthy AI, sustainability, privacy, democratization of computing power        │
│  ([Source](https://blog.google/innovation-and-ai/sundar-pichai-io-2026/)).                                      │
│    - Pichai emphasized “ambient intelligence” and making every experience smarter, safer, and more accessible.  │
│  - **Master Index of Announcements**                                                                            │
│    - Google published an official list of 100+ launches—spanning AI, hardware, developer tools, international   │
│  efforts                                                                                                        │
│  ([Source](https://blog.google/innovation-and-ai/techno

╭──────────────────────────────────────────────── Task Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: 305b26ec-c341-41ac-9f99-3c7127f68882                                                                     │
│  Agent: AI Podcast Reporting Analyst                                                                            │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 294b45e5-2d05-46a8-a988-78a4a5cf965f                                                                       │
│  Tool Args:                                                                                                     │
│  Final Output: # Podcast Briefing Report: “Google I/O 2026 – The Year Google Went ‘Agentic’: Top Announcements  │
│  and What They Mean for You”                                                                                    │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ## Episode Angle                                                                                               │
│                                                                                                                 │
│  **“Google I/O 2026: The Leap Toward Agentic Apps and Everyday AI”**                                            │
│                                                                                                                 │
│  This year’s I/O wasn’t just about new gadgets or features—it signaled Google’s pivot to deeply integrated AI   │
│  agents, “ambient intelligence,” and a future where apps aren’t just tools, but partners in getting things      │
│  done. The announcement spread from flashy new hardware to subtle but major shifts in software, privacy, and    │
│  AI reliability.                                                                                                │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ## Key Sections & Thematic Groupings                                                                           │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ### 1. Google’s Vision & Core Themes                                                                           │
│                                                                                                                 │
│  - **Sundar Pichai’s Keynote Framing**                                                                          │
│    - Google’s “north stars”: Trustworthy AI, sustainability, privacy, democratization of computing power        │
│  ([Source](https://blog.google/innovation-and-ai/sundar-pichai-io-2026/)).                                      │
│    - Pichai emphasized “ambient intelligence” and making every experience smarter, safer, and more accessible.  │
│  - **Master Index of Announcements**                                                                            │
│    - Google published an official list of 100+ launches—spanning AI, hardware, developer tools, international   │
│  efforts                                              

╭─────────────────────────────────────────── Trace Batch Finalization ────────────────────────────────────────────╮
│ ✅ Trace batch finalized with session ID: 6ee4aa6c-5ea3-42d1-83a9-2ad6491ba1bf                                  │
│                                                                                                                 │
│ 🔗 View here:                                                                                                   │
│ https://app.crewai.com/crewai_plus/ephemeral_trace_batches/6ee4aa6c-5ea3-42d1-83a9-2ad6491ba1bf?access_code=TRA │
│ CE-92c90d1dc6                                                                                                   │
│ 🔑 Access Code: TRACE-92c90d1dc6                                                                                │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

CrewOutput(raw='# Podcast Briefing Report: “Google I/O 2026 – The Year Google Went ‘Agentic’: Top Announcements and What They Mean for You”\n\n---\n\n## Episode Angle\n\n**“Google I/O 2026: The Leap Toward Agentic Apps and Everyday AI”**\n\nThis year’s I/O wasn’t just about new gadgets or features—it signaled Google’s pivot to deeply integrated AI agents, “ambient intelligence,” and a future where apps aren’t just tools, but partners in getting things done. The announcement spread from flashy new hardware to subtle but major shifts in software, privacy, and AI reliability.\n\n---\n\n## Key Sections & Thematic Groupings\n\n---\n\n### 1. Google’s Vision & Core Themes\n\n- **Sundar Pichai’s Keynote Framing**\n  - Google’s “north stars”: Trustworthy AI, sustainability, privacy, democratization of computing power ([Source](https://blog.google/innovation-and-ai/sundar-pichai-io-2026/)).\n  - Pichai emphasized “ambient intelligence” and making every experience smarter, safer, and more accessi

---

## Step 5: Swap the Topic

The `{topic}`, `{current_month}`, and `{current_year}` placeholders in agent roles and task descriptions are filled at runtime. Change the inputs and you get a different episode.


In [ ]:
# Uncomment to run a second episode on a different topic.
# podcast_crew.kickoff(inputs={
#     "topic": "AI agents for software engineering",
#     "current_month": datetime.now().strftime("%B"),
#     "current_year": str(datetime.now().year),
# })


---

## Recap

| Step | You Learned | CrewAI API |
|------|-------------|------------|
| 1 | Map your hand-rolled agent to CrewAI | `Agent(...)`, `Task(...)`, `EXASearchTool()` |
| 2 | Bound agent behavior | `max_iter`, `max_rpm`, `max_execution_time` |
| 3 | Pass one agent's output to the next | `context=[research_task]` |
| 4 | Orchestrate a multi-agent pipeline | `Crew(..., process=Process.sequential)` |
| 5 | Parametrize for reuse | `{topic}`, `kickoff(inputs=...)` |

**Next:** Open `02_production_features.ipynb` to add structured output, guardrails, and the scriptwriter agent.
